# Preparing Text Data for GPT Training

Models like GPT are mathematical machines; they understand numbers, not letters. Before we can train our model, we must first translate our human-readable text into a numerical format the model can process. By the end of this notebook, you will understand how to build a data preparation pipeline that tokenizes raw text, converts it into numerical sequences, and saves it in a format optimized for efficient training.

In [ ]:
import numpy as np
import graphviz
import os

# Set a seed for reproducibility
np.random.seed(42)

### The Core Idea: From a Novel to a Recipe

Imagine you're a chef who can only read recipes written as a sequence of numbers, where each number corresponds to a specific ingredient. You've just been handed a beautiful novel, "Romeo and Juliet," and you're told to "cook" something similar. What's the first thing you must do? You can't just throw the book in the pot!

1.  **Create an Ingredient List (Vocabulary):** First, you'd go through the entire book and list every unique character used: 'a', 'b', 'c', ' ', '.', ':', etc. This list is your "vocabulary" of ingredients.
2.  **Write the Recipe Key (Mapping):** Next, you assign a unique number to each ingredient. For example, 'a' -> 1, 'b' -> 2, and so on. This mapping is your recipe key, allowing you to translate between characters and numbers.
3.  **Transcribe the Book (Tokenization & Encoding):** Finally, you go through the book from start to finish and replace every character with its corresponding number. The sentence "Hello." might become `[8, 5, 12, 12, 15, 27]`.

This numbered sequence is exactly what a GPT model needs. Our data preparation process is this transcription process: we take a large body of raw text, create a vocabulary, and convert the entire text into a long sequence of numerical IDs, or "tokens." This numerical recipe is what we'll feed to our model, one chunk at a time.

In [ ]:
def prepare_data(raw_text, train_split_ratio=0.9):
    """
    Prepares raw text for GPT training.
    
    Args:
        raw_text (str): The input text data.
        train_split_ratio (float): The proportion of data to use for training.
        
    Returns:
        tuple: A tuple containing:
            - train_ids (np.array): The training data as a sequence of token IDs.
            - val_ids (np.array): The validation data as a sequence of token IDs.
            - int_to_char (dict): A mapping from token IDs to characters.
            - char_to_int (dict): A mapping from characters to token IDs.
    """
    # 1. Create the vocabulary
    chars = sorted(list(set(raw_text)))
    vocab_size = len(chars)
    
    # 2. Create the mappings (the "recipe key")
    char_to_int = {ch: i for i, ch in enumerate(chars)}
    int_to_char = {i: ch for i, ch in enumerate(chars)}
    
    # 3. Encode the entire dataset (the "transcription")
    all_token_ids = [char_to_int[ch] for ch in raw_text]
    data = np.array(all_token_ids, dtype=np.uint16) # Use a memory-efficient type
    
    # 4. Split into training and validation sets
    split_index = int(train_split_ratio * len(data))
    train_ids = data[:split_index]
    val_ids = data[split_index:]
    
    return train_ids, val_ids, int_to_char, char_to_int, vocab_size

### Walking Through the Implementation

Let's break down our `prepare_data` function line-by-line.

**Step 1: Create the Vocabulary**
```python
chars = sorted(list(set(raw_text)))
vocab_size = len(chars)
```
We first find all the unique characters in our text by converting the string to a `set`, which automatically handles duplicates. We then sort them to ensure our mapping is consistent every time we run the code. The total number of unique characters is our `vocab_size`.

**Step 2: Create the Mappings**
```python
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}
```
Here, we build our "recipe key." We create two dictionaries: one for encoding (character to integer) and one for decoding (integer to character). The decoding map, `int_to_char`, will be very useful later when we want to see the model's generated text.

**Step 3: Encode the Dataset**
```python
all_token_ids = [char_to_int[ch] for ch in raw_text]
data = np.array(all_token_ids, dtype=np.uint16)
```
This is the core transcription step. We iterate through the entire `raw_text` and use our `char_to_int` map to convert it into a long list of integer IDs. We then convert this list into a NumPy array, specifying `np.uint16` as the data type. This is a memory optimization: a 16-bit unsigned integer can hold values up to 65,535, which is more than enough for our character-level vocabulary, and it uses less memory than a standard Python integer or a 64-bit NumPy integer.

**Step 4: Split the Data**
```python
split_index = int(train_split_ratio * len(data))
train_ids = data[:split_index]
val_ids = data[split_index:]
```
We never want to evaluate our model on the same data it was trained on; that would be like giving a student an exam with the exact same questions they studied in class. We reserve a small portion of the data (here, 10%) as a "validation set." The model never trains on this data; we only use it to check how well the model is generalizing to new, unseen text.

In [ ]:
# --- Demonstration ---
# Let's use a simple piece of text to see our function in action.
input_text = "The quick brown fox jumps over the lazy dog."

# Prepare the data
train_data, val_data, i2c, c2i, vocab_size = prepare_data(input_text)

# --- Inspect the results ---
print(f"Vocabulary size: {vocab_size}")
print("\nCharacter-to-Integer mapping (first 5):")
for char, i in list(c2i.items())[:5]:
    print(f"  '{char}' -> {i}")

print("\nInteger-to-Character mapping (first 5):")
for i, char in list(i2c.items())[:5]:
    print(f"  {i} -> '{char}'")

print(f"\nOriginal text snippet: '{input_text[:10]}'")
print(f"Encoded token IDs: {train_data[:10]}")

print(f"\nDecoded token IDs: '{''.join([i2c[i] for i in train_data[:10]])}'")

print(f"\nTrain data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"First 5 validation tokens: {val_data[:5]}")

### Going Deeper: Efficiently Storing and Loading Data

In our simple example, the data fits easily in memory. But what happens when we're dealing with a dataset like OpenWebText, which is tens of gigabytes? We can't load it all into RAM at once.

The `nanoGPT` repository solves this by saving the prepared token ID arrays directly to binary files.
```python
train_ids.tofile('train.bin')
val_ids.tofile('val.bin')
```
This is incredibly efficient. There's no formatting, no text conversion—just a raw stream of numbers written to disk.

The real magic happens when we load the data for training. Instead of reading the entire file back into memory, we use **memory mapping**. A memory-mapped file allows the operating system to map a file on disk to a virtual memory address space. It's like having a massive NumPy array that *looks* like it's in RAM, but its contents are actually still on the disk. The OS only loads small chunks into physical RAM as they are requested.

This gives us the best of both worlds: we can work with huge datasets as if they were in memory, but without the massive RAM footprint.

In [ ]:
# Let's simulate this with our small training dataset.

# 1. Save our training data to a binary file
train_data.tofile('train.bin')

# 2. Load it back using memory-mapping
#    The OS will handle loading data into RAM only when we access it.
#    For a small file, the difference is negligible, but for a 40GB file,
#    this is the difference between the program running and crashing.
memmapped_train_data = np.memmap('train.bin', dtype=np.uint16, mode='r')

# 3. We can slice it just like a regular NumPy array!
print("Original train data (from memory):")
print(train_data[:20])

print("\nMemory-mapped train data (from disk):")
print(memmapped_train_data[:20])

# Clean up the dummy file
os.remove('train.bin')

This memory-mapping strategy is a key reason `nanoGPT` can handle massive datasets so gracefully. It's a simple but powerful technique borrowed from high-performance computing.

In [ ]:
# --- Visualization of the Data Pipeline ---

dot = graphviz.Digraph('DataPreparationPipeline', comment='The flow of data from raw text to binary files')
dot.attr(rankdir='LR', splines='ortho') # Left-to-right layout, straight-line connectors

# Define styles for nodes
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightblue')

# Create nodes
dot.node('raw_text', 'Raw Text\n("The quick brown fox...")')
dot.node('tokenize', 'Tokenization & Encoding\n(char -> int)')
dot.node('ids', 'Numerical IDs\n([1, 15, 8, ...])')
dot.node('split', 'Train/Validation Split')

dot.attr('node', shape='cylinder', fillcolor='lightgrey') # Different shape for storage
dot.node('train_bin', 'train.bin')
dot.node('val_bin', 'val.bin')


# Create edges to show the flow
dot.edge('raw_text', 'tokenize')
dot.edge('tokenize', 'ids')
dot.edge('ids', 'split')
dot.edge('split', 'train_bin', label='90%')
dot.edge('split', 'val_bin', label='10%')

# Render the graph
dot

### Summary

**What we built:** We've created a complete data preparation pipeline. It takes raw text, builds a character-level vocabulary, converts the text to numerical token IDs, and splits it into training and validation sets, ready for the model.

**Key Takeaways:**
*   **Models Need Numbers:** Language models don't understand text directly. The first step is always to convert text into a numerical representation (tokens).
*   **Vocabulary is the Key:** The vocabulary defines the universe of tokens the model knows. The mapping from token-to-integer and integer-to-token is essential for encoding data and decoding model outputs.
*   **Train/Val Split is Non-Negotiable:** Always separate your data to get an honest measure of your model's performance on unseen examples.
*   **Efficiency Matters at Scale:** For large datasets, simple choices like using memory-efficient data types (`np.uint16`) and memory-mapped files can make the difference between a feasible and an impossible project.

**What's Next:**
Our data is now perfectly prepared, like ingredients chopped and organized for a master chef. In the next notebook, we'll fire up the stove and begin cooking. We will learn how to implement the training loop that feeds this data to our GPT model, calculates loss, and updates the model's parameters to learn the patterns in the text.